In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("LargeScaleDataProcessing")
    .getOrCreate()
)

print("Spark session started successfully! 🚀")

Spark session started successfully! 🚀


In [3]:
data = [
    (1001, "Laptop", "Electronics", 1, 60000),
    (1002, "Phone", "Electronics", 2, 25000),
    (1003, "Headphones", "Audio", 3, 2000),
    (1004, "Keyboard", "Accessories", 1, 1500),
    (1005, "Mouse", "Accessories", 2, 800),
    (1006, "Monitor", "Electronics", 2, 15000),
    (1007, "Speaker", "Audio", 1, 3500),
    (1008, "Webcam", "Accessories", 2, 4500),
    (1009, "Tablet", "Electronics", 1, 30000),
    (1010, "Microphone", "Audio", 2, 5000)
]

columns = [
    "Order_ID",
    "Product",
    "Category",
    "Quantity",
    "Price"
]

sales_df = spark.createDataFrame(data, columns)

sales_df.show()

+--------+----------+-----------+--------+-----+
|Order_ID|   Product|   Category|Quantity|Price|
+--------+----------+-----------+--------+-----+
|    1001|    Laptop|Electronics|       1|60000|
|    1002|     Phone|Electronics|       2|25000|
|    1003|Headphones|      Audio|       3| 2000|
|    1004|  Keyboard|Accessories|       1| 1500|
|    1005|     Mouse|Accessories|       2|  800|
|    1006|   Monitor|Electronics|       2|15000|
|    1007|   Speaker|      Audio|       1| 3500|
|    1008|    Webcam|Accessories|       2| 4500|
|    1009|    Tablet|Electronics|       1|30000|
|    1010|Microphone|      Audio|       2| 5000|
+--------+----------+-----------+--------+-----+



In [4]:
from pyspark.sql.functions import col

sales_df = sales_df.withColumn(
    "Total_Sales",
    col("Quantity") * col("Price")
)

sales_df.show()

+--------+----------+-----------+--------+-----+-----------+
|Order_ID|   Product|   Category|Quantity|Price|Total_Sales|
+--------+----------+-----------+--------+-----+-----------+
|    1001|    Laptop|Electronics|       1|60000|      60000|
|    1002|     Phone|Electronics|       2|25000|      50000|
|    1003|Headphones|      Audio|       3| 2000|       6000|
|    1004|  Keyboard|Accessories|       1| 1500|       1500|
|    1005|     Mouse|Accessories|       2|  800|       1600|
|    1006|   Monitor|Electronics|       2|15000|      30000|
|    1007|   Speaker|      Audio|       1| 3500|       3500|
|    1008|    Webcam|Accessories|       2| 4500|       9000|
|    1009|    Tablet|Electronics|       1|30000|      30000|
|    1010|Microphone|      Audio|       2| 5000|      10000|
+--------+----------+-----------+--------+-----+-----------+



In [5]:
from pyspark.sql.functions import col, count, when, sum

# Check for missing values
sales_df.select(
    [count(when(col(c).isNull(), c)).alias(c)
     for c in sales_df.columns]
).show()

# Check for invalid quantities
invalid_quantity = sales_df.filter(
    col("Quantity") <= 0
).count()

# Check for invalid prices
invalid_price = sales_df.filter(
    col("Price") <= 0
).count()

# Check for duplicate orders
duplicate_orders = (
    sales_df.groupBy("Order_ID")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Invalid quantities:", invalid_quantity)
print("Invalid prices:", invalid_price)
print("Duplicate orders:", duplicate_orders)

+--------+-------+--------+--------+-----+-----------+
|Order_ID|Product|Category|Quantity|Price|Total_Sales|
+--------+-------+--------+--------+-----+-----------+
|       0|      0|       0|       0|    0|          0|
+--------+-------+--------+--------+-----+-----------+

Invalid quantities: 0
Invalid prices: 0
Duplicate orders: 0


In [6]:
# Create a temporary SQL view
sales_df.createOrReplaceTempView("sales")

# Analyze revenue by category
category_revenue = spark.sql("""
    SELECT
        Category,
        COUNT(Order_ID) AS Total_Orders,
        SUM(Quantity) AS Units_Sold,
        SUM(Total_Sales) AS Total_Revenue,
        ROUND(AVG(Total_Sales), 2) AS Average_Order_Value
    FROM sales
    GROUP BY Category
    ORDER BY Total_Revenue DESC
""")

category_revenue.show()

+-----------+------------+----------+-------------+-------------------+
|   Category|Total_Orders|Units_Sold|Total_Revenue|Average_Order_Value|
+-----------+------------+----------+-------------+-------------------+
|Electronics|           4|         6|       170000|            42500.0|
|      Audio|           3|         6|        19500|             6500.0|
|Accessories|           3|         5|        12100|            4033.33|
+-----------+------------+----------+-------------+-------------------+



In [7]:
from pyspark.sql.functions import year, month, to_date, lit

# Add a sample order date
sales_df = sales_df.withColumn(
    "Order_Date",
    to_date(lit("2026-08-11"))
)

# Create year and month columns for partitioning
sales_df = sales_df.withColumn(
    "Order_Year",
    year("Order_Date")
).withColumn(
    "Order_Month",
    month("Order_Date")
)

sales_df.show()

+--------+----------+-----------+--------+-----+-----------+----------+----------+-----------+
|Order_ID|   Product|   Category|Quantity|Price|Total_Sales|Order_Date|Order_Year|Order_Month|
+--------+----------+-----------+--------+-----+-----------+----------+----------+-----------+
|    1001|    Laptop|Electronics|       1|60000|      60000|2026-08-11|      2026|          8|
|    1002|     Phone|Electronics|       2|25000|      50000|2026-08-11|      2026|          8|
|    1003|Headphones|      Audio|       3| 2000|       6000|2026-08-11|      2026|          8|
|    1004|  Keyboard|Accessories|       1| 1500|       1500|2026-08-11|      2026|          8|
|    1005|     Mouse|Accessories|       2|  800|       1600|2026-08-11|      2026|          8|
|    1006|   Monitor|Electronics|       2|15000|      30000|2026-08-11|      2026|          8|
|    1007|   Speaker|      Audio|       1| 3500|       3500|2026-08-11|      2026|          8|
|    1008|    Webcam|Accessories|       2| 4500|  

In [8]:
final_sales_df = sales_df.select(
    "Order_ID",
    "Product",
    "Category",
    "Quantity",
    "Price",
    "Total_Sales",
    "Order_Date",
    "Order_Year",
    "Order_Month"
)

final_sales_df.show()

+--------+----------+-----------+--------+-----+-----------+----------+----------+-----------+
|Order_ID|   Product|   Category|Quantity|Price|Total_Sales|Order_Date|Order_Year|Order_Month|
+--------+----------+-----------+--------+-----+-----------+----------+----------+-----------+
|    1001|    Laptop|Electronics|       1|60000|      60000|2026-08-11|      2026|          8|
|    1002|     Phone|Electronics|       2|25000|      50000|2026-08-11|      2026|          8|
|    1003|Headphones|      Audio|       3| 2000|       6000|2026-08-11|      2026|          8|
|    1004|  Keyboard|Accessories|       1| 1500|       1500|2026-08-11|      2026|          8|
|    1005|     Mouse|Accessories|       2|  800|       1600|2026-08-11|      2026|          8|
|    1006|   Monitor|Electronics|       2|15000|      30000|2026-08-11|      2026|          8|
|    1007|   Speaker|      Audio|       1| 3500|       3500|2026-08-11|      2026|          8|
|    1008|    Webcam|Accessories|       2| 4500|  

In [9]:
# Save the processed Spark DataFrame as Parquet
output_path = "sales_processed_parquet"

final_sales_df.write.mode("overwrite").parquet(output_path)

print("Parquet dataset created successfully! ✅")

Parquet dataset created successfully! ✅
